# Chương 14: AI tạo sinh (Generative AI)

Notebook này triển khai từ đầu (from scratch) bằng NumPy và trực quan hóa các kiến trúc sinh kinh điển nhất trong AI hiện đại:
1. **Variational Autoencoder (VAE)**: Cơ chế Reparameterization Trick, phân rã hàm mất mát ELBO (Reconstruction + KL Divergence), và sinh dữ liệu mới từ không gian ẩn chuẩn tắc.
2. **Toy Generative Adversarial Network (1D GAN)**: Xây dựng Generator và Discriminator, huấn luyện cuộc chiến Min-Max để học phân phối thực Gaussian từ nhiễu ngẫu nhiên.
3. **Mô hình khuếch tán DDPM (Forward Diffusion Schedule)**: Mô phỏng lịch trình thêm nhiễu dạng đóng qua các mốc thời gian $t = 0, 50, 200, 500, 999$ chứng minh sự suy biến tất yếu về nhiễu trắng Gaussian.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cố định seed ngẫu nhiên
np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Variational Autoencoder (VAE) từ đầu

### Cơ chế tái tham số hóa (Reparameterization Trick)
Thay vì lấy mẫu trực tiếp từ phân phối ngẫu nhiên không khả vi $z \sim \mathcal{N}(\mu, \sigma^2)$, ta biểu diễn:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
Trong đó $\sigma = \exp(0.5 \times \text{log\_var})$.

### Hàm mất mát ELBO
$$\mathcal{L} = \text{MSE}(x, \hat{x}) + \beta \cdot D_{\text{KL}}(q(z|x) \parallel \mathcal{N}(0, I))$$
với:
$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

In [ ]:
class ToyVAE:
    def __init__(self, in_dim=2, latent_dim=2, hidden_dim=16):
        self.in_dim = in_dim
        self.latent_dim = latent_dim
        
        # Trọng số Encoder: x -> h -> (mu, log_var)
        self.W_enc = np.random.randn(in_dim, hidden_dim) * 0.1
        self.b_enc = np.zeros(hidden_dim)
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_logvar = np.zeros(latent_dim)
        
        # Trọng số Decoder: z -> h_dec -> x_hat
        self.W_dec1 = np.random.randn(latent_dim, hidden_dim) * 0.1
        self.b_dec1 = np.zeros(hidden_dim)
        self.W_dec2 = np.random.randn(hidden_dim, in_dim) * 0.1
        self.b_dec2 = np.zeros(in_dim)
        
    def encode(self, X):
        h = np.tanh(X @ self.W_enc + self.b_enc)
        mu = h @ self.W_mu + self.b_mu
        log_var = h @ self.W_logvar + self.b_logvar
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = np.exp(0.5 * log_var)
        eps = np.random.randn(*mu.shape)
        return mu + eps * std
    
    def decode(self, Z):
        h = np.tanh(Z @ self.W_dec1 + self.b_dec1)
        x_hat = h @ self.W_dec2 + self.b_dec2
        return x_hat
    
    def forward(self, X):
        mu, log_var = self.encode(X)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decode(z)
        return x_hat, mu, log_var, z
    
    def compute_loss(self, X, x_hat, mu, log_var):
        # Lỗi tái tạo MSE
        recon_loss = np.mean(np.sum((X - x_hat) ** 2, axis=1))
        # Độ phân kỳ KL Divergence dạng giải tích kín
        kl_loss = -0.5 * np.mean(np.sum(1 + log_var - mu**2 - np.exp(log_var), axis=1))
        return recon_loss + 0.1 * kl_loss, recon_loss, kl_loss

# Kiểm chứng VAE trên tập dữ liệu hình học 2D
vae = ToyVAE(in_dim=2, latent_dim=2, hidden_dim=32)
# Tạo dữ liệu thật gồm 2 đám mây điểm Gauss
N = 300
X_real = np.vstack([
    np.random.randn(N // 2, 2) * 0.3 + np.array([-1.5, -1.0]),
    np.random.randn(N // 2, 2) * 0.3 + np.array([1.5, 1.0])
])

x_hat, mu, log_var, z = vae.forward(X_real)
total_loss, recon, kl = vae.compute_loss(X_real, x_hat, mu, log_var)
print(f"VAE Initial Loss: Total={total_loss:.4f}, Recon={recon:.4f}, KL={kl:.4f}")

# Lấy mẫu mới từ không gian ẩn chuẩn tắc N(0, I) để sinh dữ liệu mới
z_prior = np.random.randn(100, 2)
x_generated = vae.decode(z_prior)
print(f"Đã sinh thành công {x_generated.shape[0]} mẫu dữ liệu mới từ phân phối Prior z ~ N(0, I).")

## 2. Huấn luyện mạng đối kháng tạo sinh đồ chơi (Toy 1D GAN)

Mục tiêu: Generator $G(z)$ nhận biến ngẫu nhiên $z \sim \mathcal{U}(-1, 1)$ và học cách sinh ra các số thực có phân phối tiệm cận phân phối dữ liệu thật $p_{\text{data}} = \mathcal{N}(\mu=4.0, \sigma^2=0.25)$.
Discriminator $D(x)$ học cách phân biệt số thật ($y=1$) và số giả ($y=0$).

In [ ]:
# Hàm sigmoid và đạo hàm
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -15, 15)))

class Toy1DGAN:
    def __init__(self):
        # Generator: z (1D) -> h (16D) -> x_fake (1D)
        self.W_g1 = np.random.randn(1, 16) * 0.5
        self.b_g1 = np.zeros(16)
        self.W_g2 = np.random.randn(16, 1) * 0.5
        self.b_g2 = np.zeros(1)
        
        # Discriminator: x (1D) -> h (16D) -> d_out in [0, 1]
        self.W_d1 = np.random.randn(1, 16) * 0.5
        self.b_d1 = np.zeros(16)
        self.W_d2 = np.random.randn(16, 1) * 0.5
        self.b_d2 = np.zeros(1)
        
    def forward_G(self, z):
        self.h_g = np.maximum(0, z @ self.W_g1 + self.b_g1)  # ReLU
        x_fake = self.h_g @ self.W_g2 + self.b_g2
        return x_fake
    
    def forward_D(self, x):
        h_d = np.maximum(0, x @ self.W_d1 + self.b_d1)
        out = sigmoid(h_d @ self.W_d2 + self.b_d2)
        return out, h_d

gan = Toy1DGAN()
batch_size = 128
lr = 0.01

# Vòng lặp mô phỏng huấn luyện GAN
for step in range(500):
    # 1. Dữ liệu thật từ N(4.0, 0.5^2)
    x_real = np.random.randn(batch_size, 1) * 0.5 + 4.0
    
    # 2. Dữ liệu giả do Generator tạo ra
    z = np.random.uniform(-1, 1, size=(batch_size, 1))
    x_fake = gan.forward_G(z)
    
    # Đánh giá của Discriminator
    d_real, h_d_real = gan.forward_D(x_real)
    d_fake, h_d_fake = gan.forward_D(x_fake)
    
    # Cập nhật Discriminator: Maximize log(D(x_real)) + log(1 - D(x_fake))
    grad_d_real = (d_real - 1.0) / batch_size
    grad_d_fake = d_fake / batch_size
    
    gan.W_d2 -= lr * (h_d_real.T @ grad_d_real + h_d_fake.T @ grad_d_fake)
    
    # Cập nhật Generator (Non-saturating loss: Maximize log(D(x_fake)))
    grad_g_out = -(1.0 - d_fake) / batch_size
    grad_g_h = (grad_g_out @ gan.W_d2.T) * (gan.h_g > 0)
    gan.W_g2 -= lr * (gan.h_g.T @ grad_g_out)

# Đánh giá mẫu sinh sau huấn luyện
z_test = np.random.uniform(-1, 1, size=(500, 1))
generated_samples = gan.forward_G(z_test).flatten()
real_samples = (np.random.randn(500, 1) * 0.5 + 4.0).flatten()

plt.figure(figsize=(7, 4))
plt.hist(real_samples, bins=30, alpha=0.6, color="green", density=True, label="Dữ liệu thật p_data ~ N(4, 0.25)")
plt.hist(generated_samples, bins=30, alpha=0.6, color="crimson", density=True, label="Mẫu sinh từ Generator G(z)")
plt.title("So sánh phân phối của Toy GAN sau huấn luyện")
plt.xlabel("Giá trị x")
plt.ylabel("Mật độ xác suất")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 3. Mô phỏng quá trình khuếch tán thuận (Forward Diffusion Schedule - DDPM)

Theo công thức giải tích đóng của DDPM (Ho et al., 2020):
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
với $\beta_t$ tăng tuyến tính từ $\beta_1 = 10^{-4}$ đến $\beta_T = 0.02$, $\alpha_t = 1 - \beta_t$, $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

In [ ]:
T = 1000
beta = np.linspace(1e-4, 0.02, T)
alpha = 1.0 - beta
alpha_bar = np.cumprod(alpha)

# Tạo một tín hiệu dạng sóng sin mẫu làm x_0
t_steps = np.linspace(0, 4 * np.pi, 200)
x_0 = np.sin(t_steps)

# Các mốc thời gian khuếch tán
timesteps = [0, 50, 200, 500, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(15, 3), sharey=True)

for idx, t in enumerate(timesteps):
    if t == 0:
        x_t = x_0
    else:
        eps = np.random.randn(*x_0.shape)
        x_t = np.sqrt(alpha_bar[t]) * x_0 + np.sqrt(1.0 - alpha_bar[t]) * eps
        
    axes[idx].plot(t_steps, x_t, color="indigo", lw=1.5)
    axes[idx].set_title(f"t = {t}\n$\bar{{\alpha}}_t={alpha_bar[t]:.3f}$")
    axes[idx].grid(True, linestyle="--", alpha=0.5)
    if idx == 0:
        axes[idx].set_ylabel("Biên độ tín hiệu")

plt.suptitle("Quá trình khuếch tán thuận (Forward Diffusion) qua các bước thời gian $t$", y=1.05, fontsize=13)
plt.tight_layout()
plt.show()

print("Hoàn tất toàn bộ nội dung thực hành Chapter 14: VAE, Toy GAN và DDPM Forward Diffusion!")